# Stronger PoliMeme classifier (ConvNeXt-Tiny finetune)

This notebook finetunes a pretrained ConvNeXt-Tiny on the PoliMemeDecode training set with class-balanced loss and threshold tuning for macro F1. It auto-detects the dataset under `/kaggle/input` and writes a `submission.csv` ready for Kaggle.

In [ ]:
# Install lightweight deps if missing
%pip install -q timm

In [ ]:
import os, random, math, time
from pathlib import Path
from typing import Optional
import numpy as np
import pandas as pd
from PIL import Image

import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader
import torch.nn.functional as F
from torchvision import transforms
from sklearn.metrics import f1_score
import timm

seed = 42
def set_seed(seed: int = 42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = False
    torch.backends.cudnn.benchmark = True
set_seed(seed)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', device)

# Auto-detect dataset root

def find_base_dir() -> Path:
    search_roots = [
        Path('/kaggle/input/polimemedecode-humor-that-speaks-politics'),
        Path('/kaggle/input'),
        Path('.')
    ]
    for root in search_roots:
        if not root.exists():
            continue
        if (root / 'Train' / 'Train.csv').exists() and (root / 'Test' / 'Test.csv').exists():
            return root
        for child in root.iterdir():
            if child.is_dir() and (child / 'Train' / 'Train.csv').exists() and (child / 'Test' / 'Test.csv').exists():
                return child
    raise FileNotFoundError('Dataset not found; set BASE_DIR manually.')

BASE_DIR = find_base_dir()
print('Using dataset at:', BASE_DIR)

TRAIN_CSV = BASE_DIR / 'Train' / 'Train.csv'
TEST_CSV = BASE_DIR / 'Test' / 'Test.csv'
TRAIN_IMG = BASE_DIR / 'Train' / 'Image'
TEST_IMG = BASE_DIR / 'Test' / 'Image'

train_df = pd.read_csv(TRAIN_CSV)
test_df = pd.read_csv(TEST_CSV)
label2id = {'NonPolitical': 0, 'Political': 1}
id2label = {v: k for k, v in label2id.items()}
print(train_df['Label'].value_counts())

In [ ]:
# Dataset + transforms
img_size = 224
mean = (0.485, 0.456, 0.406)
std = (0.229, 0.224, 0.225)

train_tfms = transforms.Compose([
    transforms.RandomResizedCrop(img_size, scale=(0.75, 1.0)),
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(0.1, 0.1, 0.1, 0.05),
    transforms.ToTensor(),
    transforms.Normalize(mean, std),
])

valid_tfms = transforms.Compose([
    transforms.Resize(int(img_size * 1.1)),
    transforms.CenterCrop(img_size),
    transforms.ToTensor(),
    transforms.Normalize(mean, std),
])

class MemeDataset(Dataset):
    def __init__(self, df: pd.DataFrame, img_dir: Path, tfm, with_labels: bool = True):
        self.df = df.reset_index(drop=True)
        self.img_dir = img_dir
        self.tfm = tfm
        self.with_labels = with_labels

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img_path = self.img_dir / row['Image_name']
        img = Image.open(img_path).convert('RGB')
        img = self.tfm(img)
        if self.with_labels:
            label = label2id[row['Label']]
            return img, label
        return img, row['Image_name']

In [ ]:
# Model + training helpers

def create_model(num_classes: int = 2):
    model = timm.create_model('convnext_tiny', pretrained=True, num_classes=num_classes, drop_path_rate=0.2)
    return model


def macro_f1_from_logits(logits: torch.Tensor, targets: torch.Tensor, thresh: float = 0.5):
    probs = logits.softmax(dim=1)[:, 1]
    preds = (probs >= thresh).long()
    return f1_score(targets.cpu().numpy(), preds.cpu().numpy(), average='macro'), probs.detach().cpu().numpy()


def evaluate(model, loader, thresh: float = 0.5):
    model.eval()
    all_probs, all_targets = [], []
    with torch.no_grad():
        for imgs, labels in loader:
            imgs = imgs.to(device)
            labels = labels.to(device)
            logits = model(imgs)
            probs = logits.softmax(dim=1)[:, 1]
            all_probs.append(probs.cpu())
            all_targets.append(labels.cpu())
    probs = torch.cat(all_probs).numpy()
    targets = torch.cat(all_targets).numpy()
    best_f1, best_t = -1.0, thresh
    for t in np.linspace(0.3, 0.7, 21):
        preds = (probs >= t).astype(int)
        f1 = f1_score(targets, preds, average='macro')
        if f1 > best_f1:
            best_f1, best_t = f1, float(t)
    return best_f1, best_t, probs, targets

In [ ]:
# Train/val split
from sklearn.model_selection import train_test_split
train_idx, val_idx = train_test_split(train_df.index, test_size=0.15, random_state=seed, stratify=train_df['Label'])
train_split = train_df.loc[train_idx].reset_index(drop=True)
val_split = train_df.loc[val_idx].reset_index(drop=True)

train_ds = MemeDataset(train_split, TRAIN_IMG, train_tfms, with_labels=True)
val_ds = MemeDataset(val_split, TRAIN_IMG, valid_tfms, with_labels=True)

def make_loader(ds, shuffle=False, bs=32):
    return DataLoader(ds, batch_size=bs, shuffle=shuffle, num_workers=2, pin_memory=True)

train_loader = make_loader(train_ds, shuffle=True, bs=32)
val_loader = make_loader(val_ds, shuffle=False, bs=32)
print(f"Train batches: {len(train_loader)}, Val batches: {len(val_loader)}")

In [ ]:
model = create_model(num_classes=2).to(device)
class_counts = train_df['Label'].value_counts()
class_weights = torch.tensor([
    1.0,
    class_counts['NonPolitical'] / class_counts['Political']
], device=device, dtype=torch.float32)
criterion = nn.CrossEntropyLoss(weight=class_weights)
optimizer = torch.optim.AdamW(model.parameters(), lr=3e-4, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=6)
scaler = torch.amp.GradScaler('cuda', enabled=torch.cuda.is_available())

best_state = None
best_f1 = -1.0
best_thresh = 0.5
epochs = 6

for epoch in range(1, epochs + 1):
    model.train()
    running_loss = 0.0
    for imgs, labels in train_loader:
        imgs = imgs.to(device)
        labels = labels.to(device)
        optimizer.zero_grad(set_to_none=True)
        with torch.amp.autocast('cuda', enabled=torch.cuda.is_available()):
            logits = model(imgs)
            loss = criterion(logits, labels)
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        running_loss += loss.item() * imgs.size(0)
    scheduler.step()

    val_f1, val_thresh, _, _ = evaluate(model, val_loader, thresh=best_thresh)
    if val_f1 > best_f1:
        best_f1 = val_f1
        best_thresh = val_thresh
        best_state = {k: v.cpu() for k, v in model.state_dict().items()}
    print(f"Epoch {epoch}: loss={running_loss/len(train_ds):.4f}, val_f1={val_f1:.4f}, best_thresh={best_thresh:.2f}")

print(f"Best val macro F1: {best_f1:.4f} at threshold {best_thresh:.2f}")
if best_state:
    model.load_state_dict(best_state)


In [ ]:
# Fit threshold on full training set using best model
full_ds = MemeDataset(train_df, TRAIN_IMG, valid_tfms, with_labels=True)
full_loader = DataLoader(full_ds, batch_size=32, shuffle=False, num_workers=2, pin_memory=True)
full_f1, tuned_thresh, _, _ = evaluate(model, full_loader, thresh=best_thresh)
best_thresh = tuned_thresh
print(f"Macro F1 on full train (for thresholding only): {full_f1:.4f}, tuned_thresh={best_thresh:.2f}")

In [ ]:
# Inference on test set
model.eval()
test_ds = MemeDataset(test_df, TEST_IMG, valid_tfms, with_labels=False)
test_loader = DataLoader(test_ds, batch_size=32, shuffle=False, num_workers=2, pin_memory=True)

all_probs = []
all_names = []
with torch.no_grad():
    for imgs, names in test_loader:
        imgs = imgs.to(device)
        logits = model(imgs)
        probs = logits.softmax(dim=1)[:, 1]
        all_probs.append(probs.cpu())
        all_names.extend(names)
probs = torch.cat(all_probs).numpy()
preds = (probs >= best_thresh).astype(int)
labels = [id2label[p] for p in preds]

submission = pd.DataFrame({
    'Image_name': all_names,
    'Label': labels,
})
submission.to_csv('submission.csv', index=False)
print(submission.head())
print('Saved submission.csv')